# routes/init

> Consolidated router assembly for the segment-align step

In [ ]:
#| default_exp routes.init

In [ ]:
#| export
from typing import List, Dict, Any, Tuple, Callable, Optional

from fasthtml.common import APIRouter, Div

from cjm_plugin_system.core.manager import PluginManager
from cjm_plugin_system.core.queue import JobQueue
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore
from cjm_fasthtml_interactions.core.state_store import get_session_id

# Sub-library route init
from cjm_transcript_segmentation.routes.init import init_segmentation_routers
from cjm_transcript_segmentation.services.segmentation import SegmentationService
from cjm_transcript_segmentation.models import SegmentationUrls
from cjm_transcript_vad_align.routes.init import init_alignment_routers
from cjm_transcript_vad_align.services.alignment import AlignmentService
from cjm_transcript_vad_align.models import AlignmentUrls

# This library's route init
from cjm_transcript_segment_align.routes.chrome import init_chrome_router
from cjm_transcript_segment_align.routes.forced_alignment import init_forced_alignment_routers
from cjm_transcript_segment_align.services.forced_alignment import ForcedAlignmentService
from cjm_transcript_segment_align.components.handlers import (
    create_seg_mutation_wrappers, build_fa_job_args, build_fa_on_complete,
    create_seg_init_chrome_wrapper, create_align_init_chrome_wrapper,
)
from cjm_transcript_segment_align.components.step_renderer import render_combined_step
from cjm_transcript_segment_align.components.keyboard_config import KB_SYSTEM_ID
from cjm_transcript_segment_align.components.sync_controls import SHOULD_PLAY_FN
from cjm_transcript_segment_align.html_ids import CombinedHtmlIds
from cjm_transcript_segment_align.models import SegmentAlignUrls, SegmentAlignResult

# Job monitor
from cjm_fasthtml_job_monitor.services.monitor import JobMonitorService
from cjm_fasthtml_job_monitor.routes.init import init_job_monitor_routes, check_inflight_job
from cjm_fasthtml_job_monitor.models import JobMonitorConfig
from cjm_fasthtml_job_monitor.components.trigger import render_job_trigger
from cjm_fasthtml_job_monitor.components.overlay import render_job_overlay_placeholder
from cjm_fasthtml_job_monitor.components.modal import get_sse_headers
from cjm_fasthtml_lucide_icons.factory import lucide_icon

## Validation

In [ ]:
#| export
def _validate_alignment(
    state:Dict[str, Any]  # Workflow state dictionary
) -> bool:  # True if segments and VAD chunks are 1:1 aligned
    """Validate 1:1 alignment between segments and VAD chunks."""
    step_states = state.get("step_states", {})
    seg_state = step_states.get("segmentation", {})
    segments = seg_state.get("segments", [])
    align_state = step_states.get("alignment", {})
    vad_chunks = align_state.get("vad_chunks", [])
    return len(segments) > 0 and len(vad_chunks) > 0 and len(segments) == len(vad_chunks)

## Init Function

In [ ]:
#| export
def init_segment_align_routers(
    state_store:SQLiteWorkflowStateStore,    # Workflow state store
    workflow_id:str,                          # Workflow identifier
    prefix:str,                               # Base prefix (e.g., "/workflow")
    source_service,                           # SourceService for fetching source blocks
    plugin_manager:PluginManager,             # Plugin manager (for services + FA)
    job_queue:JobQueue,                       # Host-owned job queue
    audio_src_url:str,                        # Host's audio serving URL
    text_plugin:str="cjm-text-plugin-nltk",           # Text plugin name
    vad_plugin:str="cjm-media-plugin-silero-vad",     # VAD plugin name
    fa_plugin_name:Optional[str]=None,        # FA plugin name (None = disabled)
    sysmon_plugin_name:Optional[str]=None,    # System monitor plugin (optional)
    max_history_depth:int=500,                # Undo history depth
) -> Tuple[List[APIRouter], SegmentAlignResult]:  # (routers, result)
    """Initialize all segment-align routers and return result bundle.

    Internally creates services, initializes sub-library routers,
    wires mutation wrappers, chrome switching, forced alignment,
    and job monitor integration.
    """
    # -----------------------------------------------------------------
    # 1. Create services
    # -----------------------------------------------------------------
    segmentation_service = SegmentationService(plugin_manager, text_plugin)
    alignment_service = AlignmentService(plugin_manager, vad_plugin)

    # -----------------------------------------------------------------
    # 2. Initialize sub-library routers
    # -----------------------------------------------------------------
    # Alignment routers (with combined init wrapper)
    wrapped_align_init = create_align_init_chrome_wrapper(should_play_fn=SHOULD_PLAY_FN)

    align_routers, align_urls, align_routes = init_alignment_routers(
        state_store=state_store,
        workflow_id=workflow_id,
        source_service=source_service,
        alignment_service=alignment_service,
        prefix=f"{prefix}/align",
        audio_src_url=audio_src_url,
        wrapped_init=wrapped_align_init,
    )

    # Segmentation routers (without mutation wrappers — need FA URLs first)
    seg_routers, seg_urls, seg_routes = init_segmentation_routers(
        state_store=state_store,
        workflow_id=workflow_id,
        source_service=source_service,
        segmentation_service=segmentation_service,
        prefix=f"{prefix}/seg",
        max_history_depth=max_history_depth,
    )

    # -----------------------------------------------------------------
    # 3. Forced alignment service + toggle route
    # -----------------------------------------------------------------
    fa_is_available = False
    fa_toggle_url = ""
    fa_routers = []
    jm_routers = []
    jm_trigger_el = None
    jm_config = None
    jm_ids = None
    jm_urls = None
    monitor_service = None
    sse_headers = []

    if fa_plugin_name:
        fa_service = ForcedAlignmentService(plugin_manager, fa_plugin_name)
        fa_service.ensure_loaded()
        fa_is_available = fa_service.is_available()

    if fa_is_available:
        # FA toggle route
        fa_router, fa_routes = init_forced_alignment_routers(
            state_store=state_store,
            workflow_id=workflow_id,
            seg_urls=seg_urls,
            prefix=f"{prefix}/fa",
        )
        fa_routers = [fa_router]
        fa_toggle_url = fa_routes["toggle"].to()

        # ----------------------------------------------------------------
        # 4. Job monitor routes for FA
        # ----------------------------------------------------------------
        monitor_service = JobMonitorService(
            queue=job_queue,
            manager=plugin_manager,
            sysmon_plugin_name=sysmon_plugin_name,
        )

        jm_config = JobMonitorConfig(
            modal_title="Force Alignment",
            trigger_label="Force Align",
            trigger_icon="audio-waveform",
        )

        jm_router, jm_urls, jm_ids = init_job_monitor_routes(
            monitor_service=monitor_service,
            plugin_name=fa_plugin_name,
            state_store=state_store,
            workflow_id=workflow_id,
            step_id="decomposition",
            state_key="fa_job_seq",
            prefix=f"{prefix}/fa-monitor",
            overlay_target_id=CombinedHtmlIds.COLUMNS,
            kb_system_id=KB_SYSTEM_ID,
            on_complete=build_fa_on_complete(
                source_service=source_service,
                seg_urls=seg_urls,
                fa_toggle_url=fa_toggle_url,
                fa_available=True,
                state_store=state_store,
                workflow_id=workflow_id,
            ),
            job_args_builder=build_fa_job_args(source_service),
            config=jm_config,
            id_prefix="fa-jm",
            icon_fn=lucide_icon,
            restore_trigger_on_complete=False,
        )
        jm_routers = [jm_router]

        # Default trigger element for FA toolbar slot (used when no job in-flight)
        jm_trigger_el = render_job_trigger(jm_config, jm_ids, jm_urls, icon_fn=lucide_icon)

        sse_headers = list(get_sse_headers())

    # -----------------------------------------------------------------
    # 5. Chrome switching router
    # -----------------------------------------------------------------
    chrome_router, chrome_routes = init_chrome_router(
        state_store=state_store,
        workflow_id=workflow_id,
        seg_urls=seg_urls,
        align_urls=align_urls,
        prefix=f"{prefix}/core/chrome",
        jm_trigger=jm_trigger_el,
        fa_toggle_url=fa_toggle_url,
        fa_available=fa_is_available,
    )
    switch_chrome_url = chrome_routes["switch_chrome"].to()

    # -----------------------------------------------------------------
    # 6. Mutation wrappers + override routes
    # -----------------------------------------------------------------
    wrapped_handlers = create_seg_mutation_wrappers(
        jm_trigger=jm_trigger_el,
        fa_toggle_url=fa_toggle_url,
        fa_available=fa_is_available,
    )

    seg_mutation_router = APIRouter(prefix=f"{prefix}/seg/workflow")

    @seg_mutation_router
    async def split(request, sess, segment_index: int):
        return await wrapped_handlers["split"](
            state_store, workflow_id, request, sess, segment_index,
            urls=seg_urls, max_history_depth=max_history_depth,
        )

    @seg_mutation_router
    async def merge(request, sess, segment_index: int):
        return await wrapped_handlers["merge"](
            state_store, workflow_id, request, sess, segment_index,
            urls=seg_urls, max_history_depth=max_history_depth,
        )

    @seg_mutation_router
    async def undo(request, sess):
        return await wrapped_handlers["undo"](
            state_store, workflow_id, request, sess, urls=seg_urls,
        )

    @seg_mutation_router
    async def reset(request, sess):
        return await wrapped_handlers["reset"](
            state_store, workflow_id, request, sess,
            urls=seg_urls, max_history_depth=max_history_depth,
        )

    @seg_mutation_router
    async def ai_split(request, sess):
        return await wrapped_handlers["ai_split"](
            state_store, workflow_id, segmentation_service, request, sess,
            urls=seg_urls, max_history_depth=max_history_depth,
        )

    # -----------------------------------------------------------------
    # 7. Seg init wrapper (combined KB system + shared chrome)
    # -----------------------------------------------------------------
    wrapped_seg_init = create_seg_init_chrome_wrapper(
        align_urls=align_urls,
        switch_chrome_url=switch_chrome_url,
        jm_trigger=jm_trigger_el,
        fa_toggle_url=fa_toggle_url,
        fa_available=fa_is_available,
    )

    seg_init_router = APIRouter(prefix=f"{prefix}/seg/workflow")

    @seg_init_router
    async def init(request, sess):
        return await wrapped_seg_init(
            state_store, workflow_id, source_service, segmentation_service,
            request, sess, urls=seg_urls,
        )

    # -----------------------------------------------------------------
    # 8. Build render closure (checks for in-flight jobs on each render)
    # -----------------------------------------------------------------
    def _render_step(ctx):
        """Pre-configured render callable for the combined step."""
        # Default job monitor elements
        trigger = jm_trigger_el
        overlay = render_job_overlay_placeholder(jm_ids) if jm_ids else None
        modal = Div(id=jm_ids.modal) if jm_ids else None
        kb_script = Div(id=jm_ids.kb_script) if jm_ids else None

        # Check for in-flight FA job (resume on reload)
        if fa_is_available and monitor_service and jm_ids and jm_urls:
            session_id = get_session_id(ctx.session) if ctx.session else None
            if session_id:
                trigger, overlay, modal, _is_running = check_inflight_job(
                    monitor_service=monitor_service,
                    plugin_name=fa_plugin_name,
                    state_store=state_store,
                    workflow_id=workflow_id,
                    session_id=session_id,
                    step_id="decomposition",
                    state_key="fa_job_seq",
                    config=jm_config,
                    ids=jm_ids,
                    urls=jm_urls,
                    icon_fn=lucide_icon,
                )

        return render_combined_step(
            ctx=ctx,
            seg_urls=seg_urls,
            align_urls=align_urls,
            switch_chrome_url=switch_chrome_url,
            fa_available=fa_is_available,
            jm_trigger=trigger,
            fa_toggle_url=fa_toggle_url,
            jm_overlay_el=overlay,
            jm_modal_el=modal,
            jm_kb_script_el=kb_script,
        )

    # -----------------------------------------------------------------
    # 9. Assemble and return
    # -----------------------------------------------------------------
    all_routers = (
        [chrome_router] + fa_routers + jm_routers +
        [seg_mutation_router, seg_init_router] +
        seg_routers + align_routers
    )

    result = SegmentAlignResult(
        urls=SegmentAlignUrls(
            seg=seg_urls,
            align=align_urls,
            switch_chrome=switch_chrome_url,
            fa_toggle=fa_toggle_url,
        ),
        render_step=_render_step,
        sse_headers=sse_headers,
        fa_available=fa_is_available,
        validate_alignment=_validate_alignment,
    )

    return all_routers, result


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()